# Import libraries

In [ ]:
import pandas as pd
import numpy as np
from os.path import join as pjoin
import matplotlib.pyplot as plt
from scipy.stats import sem
from spks.event_aligned import population_peth
from spks.utils import alpha_function
from spks.viz import plot_event_aligned_raster

pd.set_option("display.max_columns", 100)

plt.rcParams["text.usetex"] = False
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 100

%config InlineBackend.figure_format = 'svg'
%matplotlib widget
%load_ext autoreload
%autoreload 2

# Load data

In [ ]:
# %% Load data
animal = "GRB006"  # example animal
session = "20240723_142451"  # example session

data_dir = "/Users/gabriel/data"
trial_ts = pd.read_pickle(
    pjoin(data_dir, animal, session, "pre_processed", "trial_ts.pkl")
)
spike_times_per_unit = np.load(
    pjoin(data_dir, animal, session, "pre_processed", "spike_times_per_unit.npy"),
    allow_pickle=True,
)

trial_ts = trial_ts[
    trial_ts["stationary_stims"].apply(lambda x: len(x) > 0)
    & trial_ts["movement_stims"].apply(lambda x: len(x) > 0)
    & trial_ts["center_port_entries"].apply(lambda x: len(x) > 0)
].copy()

# Set up variables

In [ ]:
last_stat = (
    trial_ts["stationary_stims"].apply(lambda x: x[-1]).to_numpy()
)  # this grabs the last stim of the stationary array for every row in the pd.Series
first_move = (
    trial_ts["movement_stims"].apply(lambda x: x[0]).to_numpy()
)  # this is for grabbing the first one in the pd.Series

binwidth_ms = 10
pre_seconds = 0.1
post_seconds = 0.15

t_decay = 0.025
t_rise = 0.001
decay = t_decay / (binwidth_ms / 1000)
kern = alpha_function(
    int(decay * 15), t_rise=t_rise, t_decay=decay, srate=1.0 / (binwidth_ms / 1000)
)  # this kernel is for smoothing PETHs later

# Compare stationary vs. movement stimulus responses

## Rasters

In [ ]:
unit_id = 73

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
plot_event_aligned_raster(
    event_times=last_stat,
    spike_times=spike_times_per_unit[unit_id],
    pre_seconds=pre_seconds,
    post_seconds=post_seconds,
    ax=axs[0],
)

ymin, ymax = axs[0].get_ylim()
axs[0].vlines(x=0, ymin=ymin, ymax=ymax, colors="r", linestyles="--")
axs[0].set_title("last stationary stimulus")

plot_event_aligned_raster(
    event_times=first_move,
    spike_times=spike_times_per_unit[unit_id],
    pre_seconds=pre_seconds,
    post_seconds=post_seconds,
    ax=axs[1],
)

ymin, ymax = axs[1].get_ylim()
axs[1].vlines(x=0, ymin=ymin, ymax=ymax, colors="r", linestyles="--")
axs[1].set_title("first movement stimulus");

## PETHs

In [ ]:
def calculate_mean_sem(peth, timebin_edges, norm=False):
    if norm:
        """not currently used for these analyses"""
        mu = np.mean(
            peth[:, :, timebin_edges[:-1] < 0], axis=2, keepdims=True
        )  # mean of baseline for every unit of length trials
        sigma = np.std(peth[:, :, timebin_edges[:-1] < 0], axis=2, keepdims=True)
        sigma = np.where(sigma == 0, 1.0, sigma)
        z = (peth - mu) / sigma
        z_mean = np.mean(z, axis=1)
        z_sem = sem(z, axis=1)
        return z_mean, z_sem
    else:
        mean_peth = np.mean(peth, axis=1)
        sem_peth = sem(peth, axis=1)
        return mean_peth, sem_peth

In [ ]:
last_stat_peth, timebin_edges, event_index = population_peth(
    all_spike_times=spike_times_per_unit,
    alignment_times=last_stat,
    pre_seconds=pre_seconds,
    post_seconds=post_seconds,
    binwidth_ms=binwidth_ms,
    kernel=kern,
    pad=0,
)

last_stat_peth_mean, last_stat_peth_sem = calculate_mean_sem(
    last_stat_peth, timebin_edges, norm=False
)

first_move_peth, timebin_edges, event_index = population_peth(
    all_spike_times=spike_times_per_unit,
    alignment_times=first_move,
    pre_seconds=pre_seconds,
    post_seconds=post_seconds,
    binwidth_ms=binwidth_ms,
    kernel=kern,
    pad=0,
)

first_move_peth_mean, first_move_peth_sem = calculate_mean_sem(
    first_move_peth, timebin_edges, norm=False
)

In [ ]:
fig, ax = plt.subplots(1, figsize=(5, 5))
# ---------
# first the stationary stim
last_stat_peth_73 = last_stat_peth_mean[unit_id].copy()
last_stat_sem_73 = last_stat_peth_sem[unit_id].copy()

bin_centers = (timebin_edges[:-1] + timebin_edges[1:]) / 2.0
ax.plot(bin_centers, last_stat_peth_73, label="last stationary")
ax.fill_between(
    bin_centers,
    y1=last_stat_peth_73 - last_stat_sem_73,
    y2=last_stat_peth_73 + last_stat_sem_73,
    alpha=0.2,
)

# ---------
# now for movement stim

first_move_peth_73 = first_move_peth_mean[unit_id].copy()
first_move_sem_73 = first_move_peth_sem[unit_id].copy()

bin_centers = (timebin_edges[:-1] + timebin_edges[1:]) / 2.0
ax.plot(bin_centers, first_move_peth_73, label="first movement")
ax.fill_between(
    bin_centers,
    y1=first_move_peth_73 - first_move_sem_73,
    y2=first_move_peth_73 + first_move_sem_73,
    alpha=0.2,
)

ymin, ymax = ax.get_ylim()
ax.vlines(x=0, ymin=ymin, ymax=ymax, colors="k", linestyles="--")
ax.legend(framealpha=1, draggable=True, fontsize=10);

## Comparing stat vs. move peak responses

### Just for unit 73

In [ ]:
peak_stat = last_stat_peth.max(axis=2)  # shape (n_units, n_trials)
peak_move = first_move_peth.max(axis=2)

peak_stat_73 = peak_stat[unit_id, :]
peak_move_73 = peak_move[unit_id, :]

# plotting paired categorical points with connecting lines
fig, ax = plt.subplots(1, figsize=(3, 5))
x_positions = np.array([0, 1])

# light lines connecting each unit's pair
for s, m in zip(peak_stat_73, peak_move_73):
    ax.plot(x_positions, [s, m], color="grey", alpha=0.3, linewidth=0.8, zorder=1)

# jittered scatter so points don't sit exactly on top of the ticks
jitter = 0.1
ax.scatter(
    np.zeros_like(peak_stat_73) + np.random.normal(0, jitter, size=peak_stat_73.size),
    peak_stat_73,
    label="stationary (mean peak)",
    zorder=2,
    alpha=0.3,
)
ax.scatter(
    np.ones_like(peak_move_73) + np.random.normal(0, jitter, size=peak_move_73.size),
    peak_move_73,
    label="movement (mean peak)",
    zorder=3,
    alpha=0.3,
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["stationary", "movement"])
ax.set_ylabel("peak response (sp/s)")
ax.set_title(f"paired peak responses across trials\nunit {unit_id}", fontsize=12)
plt.tight_layout()

### For the population

In [ ]:
peak_stat = last_stat_peth.max(axis=2)  # shape (n_units, n_trials)
peak_move = first_move_peth.max(axis=2)

mean_stat = peak_stat.mean(axis=1)  # average across trials
mean_move = peak_move.mean(axis=1)

# plotting paired categorical points with connecting lines
fig, ax = plt.subplots(1, figsize=(3, 5))
x_positions = np.array([0, 1])

# light lines connecting each unit's pair
for s, m in zip(mean_stat, mean_move):
    ax.plot(x_positions, [s, m], color="0.8", linewidth=0.8, zorder=1)

# jittered scatter so points don't sit exactly on top of the ticks
jitter = 0.05
ax.scatter(
    np.zeros_like(mean_stat) + np.random.normal(0, jitter, size=mean_stat.size),
    mean_stat,
    label="stationary (mean peak)",
    zorder=2,
    alpha=0.5,
)
ax.scatter(
    np.ones_like(mean_move) + np.random.normal(0, jitter, size=mean_move.size),
    mean_move,
    label="movement (mean peak)",
    zorder=3,
    alpha=0.5,
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["stationary", "movement"])
ax.set_ylabel("peak response (sp/s)")
ax.set_title("paired peak responses per unit\ntrial averaged", fontsize=12)
plt.tight_layout()

### Another scatter plot showing the effect

In [ ]:
fig, ax = plt.subplots(1, figsize=(4, 4))

ax.scatter(
    mean_stat, mean_move, color="k", alpha=0.3
)  # NOTE: these are trial averaged peaks
min_val = min(np.min(mean_stat), np.min(mean_move))
max_val = max(np.max(mean_stat), np.max(mean_move))
ax.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.5)
ax.set_xlabel("last stationary peak (sp/s)")
ax.set_ylabel("first movement peak (sp/s)")
ax.tick_params(axis="both", which="major")
ax.set_aspect("equal")
# ax.set_xscale('log')
# ax.set_yscale('log')
fig.tight_layout()